# Ordination Methods for Ecological Data

## Overview

Ordination is the ecological term for dimensionality reduction applied to community composition data. It arranges sites (or species) in low-dimensional space so that similar sites are close together.

**Methods covered:**

| Method | Data type | Preserves | Package |
|---|---|---|---|
| PCoA (MDS) | Any distance matrix | Distances | `sklearn.MDS` |
| NMDS | Any distance matrix | Rank order of distances | `sklearn.MDS(metric=False)` |
| CA / CCA | Count data | Chi-square distances | `scikit-bio` / manual |
| RDA | Continuous response | Linear species-environment | Manual |

Ordination is distinct from PCA in ecology because species data is often sparse, zero-inflated, and described by Bray-Curtis rather than Euclidean distance.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import MDS
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform

rng = np.random.default_rng(42)
n_sites, n_species = 80, 20
assemblages = np.zeros((3, n_species))
for i in range(3):
    assemblages[i, i*6:(i+1)*6] = rng.integers(3, 10, 6)
site_type = rng.choice(3, n_sites)
X_sp = np.vstack([assemblages[t] + rng.poisson(0.8, n_species) for t in site_type]).astype(float)
env_df = pd.DataFrame({
    "elevation": [rng.normal([100,200,350][t], 30) for t in site_type],
    "nitrate":   [rng.normal([6,3,1][t],   0.8) for t in site_type],
    "ph":        [rng.normal([6.8,7.2,7.8][t], 0.2) for t in site_type]
})
print(f"Species matrix: {X_sp.shape}")
print(f"Site types: {np.bincount(site_type)}")

---
## Bray-Curtis Distance and PCoA

In [ ]:
# Bray-Curtis dissimilarity: standard for community ecology
def bray_curtis(X):
    n = X.shape[0]
    D = np.zeros((n,n))
    for i in range(n):
        for j in range(i+1,n):
            s = abs(X[i]-X[j]).sum() / (X[i].sum()+X[j].sum())
            D[i,j] = D[j,i] = s
    return D

BC = bray_curtis(X_sp)
print(f"Bray-Curtis range: [{BC.min():.3f}, {BC.max():.3f}]")
# PCoA = metric MDS on distance matrix
pcoa = MDS(n_components=2, metric=True, dissimilarity="precomputed", random_state=42)
coords = pcoa.fit_transform(BC)
fig, ax = plt.subplots(figsize=(7,5))
colors = ["steelblue","#e74c3c","#4fffb0"]
for k in range(3):
    mask = site_type == k
    ax.scatter(coords[mask,0], coords[mask,1], s=30, alpha=0.7,
               color=colors[k], label=f"Assemblage {k+1}")
ax.set_xlabel("PCoA 1"); ax.set_ylabel("PCoA 2")
ax.set_title("PCoA on Bray-Curtis Dissimilarity")
ax.legend(); plt.tight_layout(); plt.show()

---
## NMDS (Non-metric MDS)

In [ ]:
nmds = MDS(n_components=2, metric=False, dissimilarity="precomputed",
           n_init=20, random_state=42, max_iter=500)
coords_nmds = nmds.fit_transform(BC)
stress = nmds.stress_
print(f"NMDS Stress: {stress:.4f}")
print("Stress interpretation: <0.05 excellent, <0.10 good, <0.20 acceptable, >0.20 poor")
fig, ax = plt.subplots(figsize=(7,5))
for k in range(3):
    mask = site_type == k
    ax.scatter(coords_nmds[mask,0], coords_nmds[mask,1], s=30, alpha=0.7,
               color=colors[k], label=f"Assemblage {k+1}")
ax.set_xlabel("NMDS 1"); ax.set_ylabel("NMDS 2")
ax.set_title(f"NMDS (Stress={stress:.3f}) -- axes have no units")
ax.legend(); plt.tight_layout(); plt.show()

---
## Environmental Overlays

In [ ]:
# Fit environmental vectors to NMDS coordinates using correlation
fig, ax = plt.subplots(figsize=(8,6))
for k in range(3):
    mask = site_type == k
    ax.scatter(coords_nmds[mask,0], coords_nmds[mask,1], s=30, alpha=0.6,
               color=colors[k], label=f"Assemblage {k+1}")
scale = 0.4
for feat, color in zip(["elevation","nitrate","ph"], ["navy","#8B0000","darkgreen"]):
    r1, _ = np.corrcoef(coords_nmds[:,0], env_df[feat])[0]
    r2, _ = np.corrcoef(coords_nmds[:,1], env_df[feat])[0]
    # Use actual correlations
    c1 = np.corrcoef(env_df[feat], coords_nmds[:,0])[0,1]
    c2 = np.corrcoef(env_df[feat], coords_nmds[:,1])[0,1]
    ax.arrow(0, 0, c1*scale, c2*scale, head_width=0.02, head_length=0.015,
             fc=color, ec=color, lw=2)
    ax.text(c1*scale*1.2, c2*scale*1.2, feat, color=color, fontsize=10, ha="center")
ax.set_xlabel("NMDS 1"); ax.set_ylabel("NMDS 2")
ax.set_title("NMDS with Environmental Vectors")
ax.legend(); plt.tight_layout(); plt.show()

---
## Stress Plot (Shepard Diagram)

In [ ]:
# Verify NMDS: original distances vs ordination distances should be monotone
orig_dist = BC[np.triu_indices(n_sites, 1)]
ord_dist = squareform(np.sqrt(np.sum((coords_nmds[:,None]-coords_nmds[None,:])**2, axis=2)))[np.triu_indices(n_sites,1)]
rho, _ = np.corrcoef(orig_dist, ord_dist)[0]
plt.figure(figsize=(6,5))
plt.scatter(orig_dist, ord_dist, alpha=0.3, s=5, color="steelblue")
plt.xlabel("Bray-Curtis dissimilarity")
plt.ylabel("NMDS ordination distance")
plt.title(f"Shepard Diagram (r={np.corrcoef(orig_dist,ord_dist)[0,1]:.3f})")
plt.tight_layout(); plt.show()

---

## Common Pitfalls

**1. Using Euclidean distance on species abundance data**  
Euclidean distance is a poor choice for community data — two sites sharing no species may have a smaller Euclidean distance than two sites with identical species composition but different abundances. Use Bray-Curtis for abundance data and Jaccard for presence/absence.

**2. Not reporting and checking NMDS stress**  
NMDS stress measures how well the ordination preserves rank-order distances. Stress > 0.20 indicates a poor representation. Always report stress and run a Shepard diagram to visually confirm the monotone relationship.

**3. Running NMDS with a single random start**  
NMDS is iterative and can converge to local minima. Always use `n_init >= 20` and take the solution with the lowest stress.

**4. Interpreting NMDS axis direction or scale**  
NMDS axes are arbitrary — they can be rotated, reflected, or rescaled without affecting the result. Never interpret "high on NMDS 1" as meaning anything specific without overlaying environmental vectors or species scores.

**5. Applying ordination to raw counts without considering transformation**  
For highly skewed count data, consider Hellinger transformation (divide by row sum, take square root) before PCA/RDA, or use Bray-Curtis-based ordination directly. Raw counts give undue influence to the most abundant species.
---
*python_methods_library - Samantha McGarrigle*